In [22]:
##### Script to train RF model 

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import rasterio
from shapely.geometry import box
import geopandas as gpd

In [23]:
# Get the current working directory
cd = Path.cwd().parent.parent

In [24]:
##### MANUAL SET-UP

### Model details

model = 'capital' 
target = 'log_capital_intensity_USD_per_USD' 
# log_labor_intensity_jobs_per_million_USD, log_labor_intensity_jobs_per_tonne
# log_capital_intensity_USD_per_USD, log_capital_intensity_USD_per_tonne


In [25]:
##### Model set-up (automatic)

# load data
model_data = pd.read_csv(f'{cd}/Data/Clean/Training_data/{model}_final.csv')

# # load CV folds 
# folds = pd.read_csv(f'{cd}/Data/Fold_assignments/{model}_folds.csv')

# set predictors
predictors = [
       'SOC', 'pct_cropland_irrigated', 'female_share',
       'pop_density_people_per_km2', 'average_travel_time_city',
       'average_travel_time_port', 'probability_economic_land_use_objective',
       'probability_survival_land_use_objective', 'growing_season_length_days',
       'child_dependency_ratio', 'GDP_pc', 'slope',
       'cereals_share_production_USD', 'fibres_share_production_USD',
       'fruits_share_production_USD', 'oilcrops_share_production_USD',
       'pulses_share_production_USD', 'roots_tubers_share_production_USD',
       'rest_of_crops_share_production_USD',
       'sugar_crops_share_production_USD', 'vegetables_share_production_USD',
       'rubber_share_production_USD', 'ruminants_share_production_USD',
       'monogastrics_share_production_USD', 'poultry_share_production_USD',
       'pct_GDP_ag', 'share_large_field', 'share_medium_field',
       'share_small_field', 'share_vsmall_field', 'lon', 'lat',
       'share_with_nightlights', 'crop_intensity', 
       'log_country_capital_intensity_USD_per_USD',
       'log_country_labor_intensity_jobs_per_million_USD'
]

# set parameters for grid-search 
grid_search = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_leaf': [1, 2, 5, 10],
    'max_features': ['sqrt', 0.5, 0.75],
}

In [26]:
##### Set spatial blocks for CV
# 10 blocks = 5 states per block, for a 80/20 split 

n_folds = 10
countries = model_data['country_ID'].unique()

np.random.seed(31)
np.random.shuffle(countries)

country_folds = np.array_split(countries, n_folds)  

In [27]:
# set lists to store results
metrics = []
feature_importances = []
models = []

# run model for each fold, including grid search for each 
for fold_idx, test_countries in enumerate(country_folds):
    print(f"Fold {fold_idx + 1}/{n_folds}")
    
    # split train and test for fold
    train_mask = ~model_data['country_ID'].isin(test_countries)
    test_mask  =  model_data['country_ID'].isin(test_countries)

    X_train = model_data.loc[train_mask, predictors]
    y_train = model_data.loc[train_mask, target]
    X_test  = model_data.loc[test_mask,  predictors]
    y_test  = model_data.loc[test_mask,  target]

    # train model - doing grid search for each fold
    # uses CV within each train dataset to determine 'best' model
    rf = RandomForestRegressor(random_state=27, n_jobs=-1)

    search = RandomizedSearchCV(
        rf, grid_search,
        n_iter=20,
        cv=5,
        scoring='neg_root_mean_squared_error',
        random_state=17,
        n_jobs=-1
    )

    search.fit(X_train, y_train)
    best_model = search.best_estimator_

    # evaluate models based on test data
    y_train_pred = best_model.predict(X_train)
    y_test_pred = best_model.predict(X_test)
    train_r2     = r2_score(y_train, y_train_pred)
    test_r2      = r2_score(y_test,  y_test_pred)

    metrics.append({
        'fold':       fold_idx + 1,
        'best_params': search.best_params_,
        'RMSE':       np.sqrt(mean_squared_error(y_test, y_test_pred)),
        'MAE':        mean_absolute_error(y_test, y_test_pred),
        'R2':         r2_score(y_test, y_test_pred),
        'n_train':    len(y_train),
        'n_test':     len(y_test),
    })

    feature_importances.append(best_model.feature_importances_)
    models.append(best_model)
    
    print(f"  Train R²: {train_r2:.4f}  Test R²: {test_r2:.4f}")

Fold 1/10
  Train R²: 0.9433  Test R²: -0.6550
Fold 2/10


KeyboardInterrupt: 